In [1]:
!pip install tensorflow tensorflow-hub streamlit pandas numpy groq -q
!npm install localtunnel -q

⠙⠹⠸⠼⠴⠦⠧
up to date, audited 23 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠇

In [2]:
import pandas as pd
import json
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, regularizers, callbacks
import tensorflow_hub as hub

# I have used this dataset from kaggle
df = pd.read_csv("software_requirements_extended.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

unique_labels = df['Type'].unique()
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
df['label'] = df['Type'].map(label_mapping)

with open("label_mapping.json", "w") as f:
    json.dump({int(v): k for k, v in label_mapping.items()}, f)

texts = df['Requirement'].astype(str).values
labels = df['label'].values
num_classes = len(unique_labels)


model = keras.Sequential([

    hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4",
                   input_shape=[], dtype=tf.string, trainable=False),


    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(num_classes, activation='softmax')
])


model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)


model.fit(texts, labels, epochs=50, batch_size=16, validation_split=0.15, callbacks=[early_stop])

model.save("tf_hub_model.keras")


Epoch 1/50
52/52 [==============================] - 13s 74ms/step - loss: 2.5244 - accuracy: 0.3000 - val_loss: 2.4163 - val_accuracy: 0.2449
Epoch 2/50
52/52 [==============================] - 2s 39ms/step - loss: 1.3475 - accuracy: 0.5988 - val_loss: 2.2392 - val_accuracy: 0.3197
Epoch 3/50
52/52 [==============================] - 2s 41ms/step - loss: 0.8799 - accuracy: 0.7398 - val_loss: 2.0697 - val_accuracy: 0.4490
Epoch 4/50
52/52 [==============================] - 3s 52ms/step - loss: 0.7162 - accuracy: 0.7880 - val_loss: 1.8938 - val_accuracy: 0.4694
Epoch 5/50
52/52 [==============================] - 2s 39ms/step - loss: 0.6414 - accuracy: 0.8133 - val_loss: 1.7033 - val_accuracy: 0.4966
Epoch 6/50
52/52 [==============================] - 2s 38ms/step - loss: 0.4893 - accuracy: 0.8735 - val_loss: 1.5118 - val_accuracy: 0.5306
Epoch 7/50
52/52 [==============================] - 2s 40ms/step - loss: 0.4349 - accuracy: 0.8675 - val_loss: 1.3155 - val_accuracy: 0.5714
Epoch 8/50
5

In [15]:
%%writefile app.py
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import streamlit as st
import tensorflow as tf
import tf_keras as keras
import tensorflow_hub as hub
import numpy as np
import json
from groq import Groq


GROQ_API_KEY = "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"

@st.cache_resource
def load_labels():
    with open("label_mapping.json", "r") as f:
        return json.load(f)

@st.cache_resource
def load_my_model():
    return keras.models.load_model(
        "tf_hub_model.keras",
        custom_objects={'KerasLayer': hub.KerasLayer}
    )

def generate_code_via_llm(requirement, category):

    try:
        client = Groq(api_key=GROQ_API_KEY)


        prompt = f"""
        You are a Senior QA Automation Engineer.
        Task: Write a Python 'pytest' test suite for the following {category} requirement.
        Requirement: '{requirement}'

        Instructions:
        1. Include a test for the 'Happy Path' (valid input).
        2. Include at least two edge cases or failure scenarios.
        3. Use mock data or comments to show where real data would go.
        4. Output ONLY the raw Python code. Do not include markdown blocks or explanations.
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"# Groq API Error: Make sure your key is valid and you are connected to the internet.\n# Error Details: {e}"


st.set_page_config(page_title="AI QA Agent", page_icon="🧪", layout="centered")
st.title("🧪 Smart QA Automation Agent")
st.markdown("Powered by **TF-Hub Universal Sentence Encoder** & **Llama-3**.")

try:
    labels_map = load_labels()
    model = load_my_model()

    user_input = st.text_area(
        "Paste an enterprise software requirement:",
        placeholder="e.g., The system must lock the user account after 5 consecutive failed login attempts."
    )

    if st.button("Analyze & Build Test Suite", type="primary"):
        if user_input.strip():
            with st.spinner("TensorFlow Hub mapping semantic architecture..."):


                predictions = model.predict(np.array([user_input]))
                predicted_idx = int(np.argmax(predictions[0]))
                confidence = np.max(predictions[0]) * 100
                category_name = labels_map.get(str(predicted_idx), "Unknown")

            st.success(f"🧠 **TensorFlow Result:** Categorized as `{category_name}` ({confidence:.1f}% confidence)")

            with st.spinner("Llama-3 synthesizing pytest scenarios..."):


                code_output = generate_code_via_llm(user_input, category_name)
                st.subheader("Generated Test Suite")
                st.code(code_output, language="python")


        else:
            st.warning("Please type a requirement first.")
except Exception as e:
    st.error(f"Error loading system: {e}")

Overwriting app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &


!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

print("🌐 YOUR LIVE APP LINK WILL APPEAR BELOW 🌐")
print("👉 Look for the link that ends in '.trycloudflare.com'")
!./cloudflared tunnel --url http://localhost:8501

Starting Streamlit...
🌐 YOUR LIVE APP LINK WILL APPEAR BELOW 🌐
👉 Look for the link that ends in '.trycloudflare.com'
2026-05-23T10:52:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-23T10:52:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-23T10:52:59Z INF +--------------------------------------------------------------------------------------------+
2026-05-23T10:52:59Z INF |  Your quick Tunnel has been created! Visit it at (it may 